# Pump house feed: an Integration, end to end

A pretend pump house writes sensor readings to a CSV file. A real **Cognite DB extractor** reads that file every 15 seconds and sends the rows to CDF, where a workflow turns them into assets and time series. You run it all from this notebook, then break it on purpose to see what monitoring shows.

Run the cells from top to bottom. Each step ends with **✅ Verify in Cognite Data Fusion**, which tells you what to look for.

| Step | What you do |
|---|---|
| 1 - 3 | Connect, allow Integrations, deploy everything |
| 4 | Deploy the monitoring app, to watch everything from here on |
| 5 | Start the extractor |
| 6 - 7 | Feed data in and see it arrive |
| 8 - 9 | Break the data, then break the extractor's configuration |
| 10 | Stop everything and clean up |

## Step 1 - Connect

Uses the credentials in the `.env` file in the root of this repository. If the file is missing, the cell stops and tells you what to do: copy the `.env` file from your sandbox image repo into this repository, next to `pyproject.toml`.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

import ipywidgets as widgets
from IPython.display import display

from helpers import extractor_config, feed, ops
from helpers.auth import get_client, require_env_file

print(f"Using {require_env_file()}")  # stops here, with instructions, if there is no .env file
client = get_client()
print(f"Connected to CDF project '{client.config.project}'")


def make_log(height):
    """A small log box. Returns (widget, write) - call write(text) to add a line."""
    box = widgets.Textarea(disabled=True, layout={"width": "100%", "height": height})

    def write(text):
        box.value += f"{text}\n"

    return box, write


def show_status():
    """One-glance summary of the whole pipeline. Re-run it after any step."""
    extractor = ops.extractor_status(client)
    state = "connected" if extractor["connected"] else "NOT connected"
    print(f"Extractor    : {state} - config revision in use {extractor['active_revision']}, latest {extractor['latest_revision']}")
    print(f"Rows in CSV  : {feed.status()['rows']}")
    print(f"Rows in RAW  : {ops.raw_row_count(client)} ({ops.raw_rows_last_minute(client)} written in the last minute)")
    print(f"Datapoints   : {sum(ops.datapoint_counts(client).values())}")
    errors = ops.get_errors(client)
    if errors:
        newest = max(errors, key=lambda error: error["startTime"])
        state = "resolved" if newest.get("endTime") else "ACTIVE"
        print(f"Latest error : {newest['type']}: {newest['description'][:100]} ({state})")
    newest_runs = {}
    for run in ops.transformation_runs(client, limit=1):
        newest_runs.setdefault(run["transformation"], run)
    for name, run in sorted(newest_runs.items()):
        detail = f" - {run['error'][:100]}" if run["error"] else ""
        print(f"{name:<24}: {str(run['status']).split('.')[-1]}{detail}")

## Step 2 - Allow Integrations

Integrations is a public-preview feature, so the service principal in your `.env` needs permission to create and run integrations. This adds just those permissions to its group and leaves everything else alone. It is safe to run again.

In [ ]:
from helpers.capabilities import grant_integration_capabilities

result = grant_integration_capabilities(client)
if result["added"]:
    print(f"Added to group '{result['group']}': {result['added']}")
else:
    print(f"Group '{result['group']}' already has the Integrations permissions.")

### ✅ Verify in Cognite Data Fusion

Open the group `cognite_toolkit_service_principal` in Fusion's access management. It should now list the capabilities `integrations` (read, write, use) and `integrationconfigs` (read, write).

## Step 3 - Deploy

This cell runs the Cognite Toolkit twice: first `cdf build`, then `cdf deploy`. Together they build the module in `modules/psp_integrations/` and deploy it to your project. It creates:

- the **integration**, with its first **configuration revision** - the settings the extractor will download: which file to read, the query, and the 15 second schedule
- a RAW table to land the rows in
- a **workflow** with three transformations that turn the rows into assets, time series and datapoints, plus the space they are written to
- a **location filter**, so the data shows up in Search

It takes about a minute.

In [ ]:
from helpers import toolkit

toolkit.build_and_deploy()

### ✅ Verify in Cognite Data Fusion

Open **Integrate → Integrations**. You should see:

- **Pump house feed** in the list. Nothing is connected to it yet.
- On its **Configuration** tab, the extractor settings: the CSV file it reads, the query, and a 15 second schedule.

Under **Data workflows** you will also find `wf_psp_pump_processing`.

## Step 4 - Monitor it

Before anything runs, deploy the monitoring app. It shows the integration's health, how fast rows arrive, its errors and the workflow runs on one page, and it refreshes every 15 seconds. Keep it open in a browser tab and glance at it after each step below. It is a **Custom Flows app**.

You can deploy it to your environment and open it from the link the deploy prints, which is enough to try it out. What you cannot do here is activate it in the **Custom apps** dashboard: that needs signatures from a certified Flows developer, and the app itself has to be certified. See [Builder certification](https://docs.cognite.com/cdf/flows/guides/builder-certification) and [Application certification](https://docs.cognite.com/cdf/flows/guides/application-certification).

To deploy it, run this from the repository root in a terminal:

```powershell
.\deploy-app.ps1
```

It opens a browser to sign in, then prints a link. Click the link to open the app.

### ✅ Verify in Cognite Data Fusion

The app opens inside Fusion, titled **Pump house feed**, with its version next to the title. Nothing is running yet, so it is quiet:

- **Connection** shows the config revision from Step 3 and *none reported yet* under Tasks.
- **Readings staged** and **Task runs** are 0, and **Feed pace** says *Nothing staged yet*.
- The badge at the top says **Healthy**.

## Step 5 - Start the extractor

The extractor runs on your own machine, like it would at a real site.

1. In Fusion open **Integrate → Integrations → Pump house feed** and follow its download instructions for the **Cognite DB extractor**. Download the latest **v4.x standalone** version (a 3.x version does not work with Integrations). Unzip it and put `dbextractor-standalone-....exe` in this repository's `extractor/` folder.
2. In a terminal, at the repository root, run:

   ```powershell
   .\run-extractor.ps1
   ```

   Leave it running. The script loads your `.env` first, because the extractor reads its credentials from environment variables and does not read `.env` itself.


Then run the cell below. It may take up to 30 seconds after you start the extractor before it shows as connected.

In [ ]:
status = ops.extractor_status(client)
if status["connected"]:
    print("Extractor connected.")
    print(f"Tasks    : {status['tasks']}")
    print(f"Revision : {status['active_revision']} in use")
else:
    print("The extractor is not connected yet. Start .\\run-extractor.ps1 in a terminal, wait a little, and run this cell again.")


> **⚠️ Expect an error in the logs for now.** The data file `data/pump_readings.csv` does not exist until you press **Start feed** in Step 6. Until then the extractor cannot find it and fails on every 15 second cycle. This is expected, so do not worry. It goes away once the feed is running.
>
> - In the terminal where the extractor runs, you will see `Unexpected error during execution for query 'pump_readings' on database 'pump_source'`, followed by `Could not read file ./data/pump_readings.csv: The system cannot find the file specified.`
> - In the monitoring app, the same *Unexpected error during execution for query 'pump_readings'* shows up under **Errors**, and the badge turns to **1 open error(s)**.
>
> Carry on to Step 6. Once the file exists, the next cycle succeeds and the error gets an **Ended** time and the state `resolved`.

### ✅ Verify in Cognite Data Fusion

Open **Integrate → Integrations → Pump house feed**. On the **Overview** tab you should see the task `pump_readings`, and the extractor as just seen.

**In the monitoring app** (reloads itself every 15 seconds): **Connection** now lists the extractor with its version, for example `cognite-db 4.0.0-beta3`, and the task `pump_readings`.

## Step 6 - Feed data in

The extractor reads `data/pump_readings.csv`. This cell is the pump house writing to it. Press **Start feed**, and change the **pace** whenever you like.

**The feed runs in the background, so you do not have to wait for it.** Leave it running and carry on with the next steps. It stops when you press **Stop feed** (or in Step 10), or when you close this notebook.

| Pace | Readings written |
|---|---|
| `paused` | none |
| `slow` | 1 per pump every 30 seconds |
| `normal` | 1 per pump every 5 seconds |
| `fast` | 5 per pump every second |
| `burst` | 200 per pump every 10 seconds |

In [ ]:
existing = feed.status()["rows"]
if existing:
    answer = input(
        f"data/pump_readings.csv already holds {existing} readings from an earlier run. "
        "Type YES to start from blank, anything else to keep them and carry on: "
    )
    if answer.strip() == "YES":
        feed.start_blank()
        print("Started from blank. If the extractor is already running, stop it (Ctrl+C) and run .\run-extractor.ps1 again, so it forgets its old position.")
    else:
        print(f"Keeping the {existing} existing readings.")
else:
    print("No earlier readings - starting from blank.")

In [ ]:
feed_log, write_feed_log = make_log("120px")
runner = feed.FeedRunner(on_event=write_feed_log)

start_button = widgets.Button(description="Start feed", button_style="success", icon="play")
stop_button = widgets.Button(description="Stop feed", button_style="warning", icon="stop")
pace_picker = widgets.ToggleButtons(options=list(feed.PACES), value="normal", description="Pace:")

start_button.on_click(lambda _: runner.start())
stop_button.on_click(lambda _: runner.stop())
pace_picker.observe(lambda change: runner.set_pace(change["new"]), names="value")

display(widgets.VBox([widgets.HBox([start_button, stop_button]), pace_picker, feed_log]))

Give it a minute, then run this cell. Run it again after you change the pace.

In [ ]:
show_status()

### ✅ Verify in Cognite Data Fusion

- **Integrate → Staging**: open `psp_integrations` → `pump_readings`. Rows arrive every 15 seconds.
- **Integrate → Integrations → Pump house feed → Task log**: a run of `pump_readings` about every 15 seconds.
- Change the pace and run the cell above again. *written in the last minute* rises and falls with the pace.

**In the monitoring app:** **Readings staged** counts up and the **Feed pace** chart fills with one bar per minute. Change the pace and the bars grow or shrink to match. **Task runs** also climbs, because the extractor reports each cycle.

## Step 7 - The workflow

A workflow, `wf_psp_pump_processing`, runs every 5 minutes. It runs three transformations one after another, which turn the rows in RAW into data in the Core Data Model:

1. `tr_psp_pump_assets` creates a **Pump house** asset, and below it an asset for each pump.
2. `tr_psp_pump_timeseries` creates the time series for each pump: flow rate, pressure and temperature.
3. `tr_psp_pump_datapoints` adds the readings to those time series.

There is nothing to run here. Give it up to 10 minutes: the first run creates the assets and time series, and the next one can add the datapoints to them.

### ✅ Verify in Cognite Data Fusion

- **Integrate → Data workflows → `wf_psp_pump_processing`**: a run every 5 minutes, each with its three tasks.
- **Integrate → Transformations**: `tr_psp_pump_assets`, `tr_psp_pump_timeseries` and `tr_psp_pump_datapoints`, each with its latest run.
- **Search**: pick the location **Pump House**. You should see the asset **Pump house** with the pumps `PUMP-101`, `PUMP-102` and `PUMP-103` below it (open the asset hierarchy to see the tree), and their time series such as `PUMP-101:flow_rate`. Open a time series: the preview shows the readings. Set the time frame to **1H**.

**In the monitoring app:** under **Downstream**, *Recent workflow executions* gets a new row every 5 minutes, and *Recent transformation runs* lists the three transformations as `Completed`. The first run may show no datapoints yet, as described above.

## Step 8 - Break the data

A healthy integration can still be handed bad data. **Inject bad data** adds a reading where the flow rate is `N/A`, as if a sensor had gone offline. The extractor cannot read the file with that value in it, so its `pump_readings` task fails on every cycle, and each failure is reported to the integration.

If the file is still short, the button first adds some normal readings. The extractor decides what type each column is from the first 100 rows, so a bad value only breaks it after that.

In [ ]:
bad_data_log, write_bad_data_log = make_log("80px")
bad_data_button = widgets.Button(description="Inject bad data", button_style="danger", icon="bug")
fix_data_button = widgets.Button(description="Remove the bad data", button_style="info", icon="wrench")


def run_data_action(action):
    write_bad_data_log(action())


bad_data_button.on_click(lambda _: run_data_action(feed.inject_bad_data))
fix_data_button.on_click(lambda _: run_data_action(feed.remove_bad_data))
display(widgets.VBox([widgets.HBox([bad_data_button, fix_data_button]), bad_data_log]))

Press **Inject bad data**, wait about 30 seconds, then run this cell:

In [ ]:
show_status()

### ✅ Verify in Cognite Data Fusion

After **Inject bad data**, open **Integrate → Integrations → Pump house feed**:

- The **Errors** tab gets a new error about every 15 seconds: *Unexpected error during execution for query 'pump_readings'*. Its details say that `N/A` could not be parsed as a number in the column `flow_rate`.
- The **Task log** shows those `pump_readings` runs as failed, and no new rows arrive in RAW, because the extractor cannot read the file.

**In the monitoring app:** the badge at the top turns to **1 open error(s)** (it grows as the failures repeat), **Open errors** counts up, and the **Errors** panel lists each failure at the top, state `active`, with an empty **Ended** column. The **Feed pace** chart goes flat, because no new rows arrive.

Now press **Remove the bad data**. The next cycle, within 15 seconds, succeeds and the errors stop. In the app, the errors get an **Ended** time and the state `resolved`, the badge goes back to **Healthy**, and the bars in **Feed pace** return.

## Step 9 - Break the extractor's configuration

The extractor keeps no settings locally. It downloads its configuration from the integration when it starts, and **checks for a newer revision every 30 seconds**. When one appears it restarts with it. That lets you control an extractor on a machine you cannot reach, from the cloud.

**Break the config** publishes a new revision with a required setting removed. **Repair the config** publishes the last good one again.

In [ ]:
config_log, write_config_log = make_log("80px")
break_button = widgets.Button(description="Break the config", button_style="danger", icon="unlink")
repair_button = widgets.Button(description="Repair the config", button_style="info", icon="wrench")


def run_config_action(action):
    try:
        write_config_log(action(client))
    except Exception as error:
        write_config_log(f"Failed: {error}")


break_button.on_click(lambda _: run_config_action(extractor_config.break_config))
repair_button.on_click(lambda _: run_config_action(extractor_config.repair_config))
display(widgets.VBox([widgets.HBox([break_button, repair_button]), config_log]))

Press **Break the config**, wait 30 to 45 seconds, then run this cell:

In [ ]:
show_status()

### ✅ Verify in Cognite Data Fusion

After **Break the config**:

- **Integrate → Integrations → Pump house feed → Configuration**: a new revision, described as *Broken on purpose*.
- Within about 45 seconds the **Errors** tab shows a `config` error: *Invalid config: Missing 'sheet' parameter…*
- The extractor refused the new settings and stopped. **Task log** gets no new runs, and rows stop arriving in RAW. The revision in use stays at the last good one.

**In the monitoring app:** **Connection** shows the new **Config revision**, and the **Errors** panel gets a `config` error (*Invalid config: Missing 'sheet' parameter…*) with state `active`. The **Feed pace** chart goes flat.

Now press **Repair the config**, wait 30 to 45 seconds, and run the cell above again. The extractor picks up the new revision, restarts, and rows start arriving again. In the app, the `config` error gets an **Ended** time and the state `resolved`, and the bars in **Feed pace** return.

## Step 10 - Stop and clean up

1. **Stop the feed.** Press **Stop feed** in Step 6, or run the cell below.
2. **Stop the extractor.** Go to the terminal where `.\run-extractor.ps1` is running and press **Ctrl+C**.
3. Open `cleanup.ipynb` and run it. It removes what this notebook deployed to your project: the integration, the workflow and transformations, the data, and the local files. It deletes nothing until you type `YES` in the box that pops up.

Stop the feed and the extractor first. A running extractor keeps checking in and writing data, so it would be working against the cleanup.

In [ ]:
runner.stop()
print("Feed stopped. Now stop the extractor with Ctrl+C in its terminal, then open cleanup.ipynb.")